# Web search tool

Claude's **built-in** web search tool. Fully Anthropic-hosted: Anthropic publishes the schema (`type="web_search_20250305"`) and executes the search on its side.
No local handler, no `run_tools` loop — just pass the schema and read citations off the response.

## Schema fields

- `type`: `web_search_20250305` (dated API version).
- `name`: `web_search`.
- `max_uses`: cap on total searches per request (default 5).
- `allowed_domains` (optional): restrict results to specific domains. Here we restrict to `nih.gov` so answers come from authoritative medical sources.

## Response

The returned message has mixed block types: `text` (Claude's answer, each with optional `citations`), `server_tool_use` (the queries Claude issued), and `web_search_tool_result` (the hits for each query).
Citations attach to text blocks and link each cited sentence to its source URL.
## Getting schemas for Anthropic tools

For user-implemented tools you write both the schema and the handler. For Anthropic's built-in tools you don't write the schema:
Anthropic publishes versioned schemas that you reference by `type` (e.g. `text_editor_20250728`, `web_search_20250305`), plus optional config fields (`max_uses`, `allowed_domains`, ...).
You pass the minimal stub and Anthropic expands the full schema internally.

Examples:
[Text editor](https://platform.claude.com/docs/en/agents-and-tools/tool-use/text-editor-tool),
[Web search](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool),
[Code execution](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool),
[Computer use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/computer-use-tool),
[Bash](https://platform.claude.com/docs/en/agents-and-tools/tool-use/bash-tool),
[Memory](https://platform.claude.com/docs/en/agents-and-tools/tool-use/memory-tool).


In [1]:
from chat_tools import add_user_message, chat
from models import MODEL_SONNET

In [2]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [3]:
messages = []
add_user_message(
    messages, "Summarize recent NIH-indexed studies on resistance training protocols for leg muscle hypertrophy."
)
response = chat(messages, model=MODEL_SONNET, tools=[web_search_schema])

for block in response.content:
    print(f"--- {block.type} ---")
    if block.type == "text":
        print(block.text)
        for citation in getattr(block, "citations", None) or []:
            print(f"  [cite] {citation.cited_text}  <- {citation.url}")
    elif block.type == "server_tool_use":
        print(f"query: {block.input}")
    elif block.type == "web_search_tool_result":
        for hit in block.content:
            print(f"- {hit.title}: {hit.url}")
    else:
        print(block)
    print()

--- server_tool_use ---
query: {'query': 'NIH PubMed resistance training leg muscle hypertrophy studies 2024 2025'}

--- server_tool_use ---
query: {'query': 'resistance training protocols lower body hypertrophy randomized controlled trial NIH 2024'}

--- web_search_tool_result ---
- The impact of resistance training on gluteus maximus hypertrophy: a systematic review and meta-analysis - PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC12018462/
- Similar muscle hypertrophy following eight weeks of resistance training to momentary muscular failure or with repetitions-in-reserve in resistance-trained individuals - PubMed: https://pubmed.ncbi.nlm.nih.gov/38393985/
- Effects of leg immobilization and recovery resistance training on skeletal muscle-molecular markers in previously resistance-trained versus untrained adults - PubMed: https://pubmed.ncbi.nlm.nih.gov/39819075/
- Resistance Training Beyond Momentary Failure: The Effects of Past‐Failure Partials Versus Initial Partials on Calf Musc